In [3]:
import pandas as pd

df = pd.read_excel("/workspaces/my-projects/database_cleaning/ireland_contacts.xlsx")

In [4]:
# Checking duplicates based on Email
df.duplicated(subset='Email').sum()

np.int64(7725)

In [5]:
# Show rows where Email is duplicated
df[df.duplicated(subset='Email', keep=False)].sort_values('Email').head(20)

,Full Name,Create Date,Email,Industry,Website URL,Country/Region
49487,(No value),2024-05-23 12:47:00,(No value),Manufacturing,www.candcgroupplc.com,Ireland
8885,(No value),2024-05-23 12:47:00,(No value),(No value),teelingwhiskey.com,Ireland
33480,(No value),2024-03-21 17:02:00,(No value),Manufacturing,arla.com,United Kingdom
10730,(No value),2024-05-23 12:47:00,(No value),Logistics,dhl.com,Ireland
32661,(No value),2024-03-21 17:02:00,(No value),Food & Beverages,eastendfoods.co.uk,United Kingdom
45285,(No value),2024-05-23 12:47:00,(No value),Dairy,dairygold.ie,Ireland
14320,(No value),2020-03-31 08:20:00,(No value),(No value),alliance-healthcare.co,Scotland
36875,(No value),2024-05-23 12:47:00,(No value),Manufacturing,valeofoods.ie,Ireland
13245,(No value),2024-05-23 12:47:00,(No value),Food Production,dawnfarms.ie,(No value)
61662,(No value),2024-05-23 12:47:00,(No value),Sporting Goods,adidas-group.com,Germany


In [6]:
df['Country/Region'].value_counts(dropna=False)

Country/Region
United Kingdom         35712
Ireland                13500
(No value)              9913
England                 2754
Scotland                2537
                       ...  
United Kindgdom            1
South Glamorgan            1
Trinidad and Tobago        1
Makati                     1
Mohali                     1
Name: count, Length: 356, dtype: int64

In [7]:
# Keep Ireland OR (.ie domains with missing country)

filtered_df = df[
    (df['Country/Region'] == 'Ireland') |
    (
        (df['Country/Region'] == '(No value)') &
        (df['Website URL'].str.endswith('.ie', na=False))
    )
]

In [8]:
filtered_df.shape

filtered_df['Country/Region'].value_counts(dropna=False)

Country/Region
Ireland       13500
(No value)      377
Name: count, dtype: int64

In [9]:
filtered_df[filtered_df['Country/Region'] == '(No value)'].head(20)

,Full Name,Create Date,Email,Industry,Website URL,Country/Region
77,(No value),2021-02-10 09:08:00,tom.dineen@kerry.ie,Food,kerry.ie,(No value)
78,(No value),2023-05-17 20:29:00,trish.mccarthy@eirdata.ie,Pharma,eirdata.ie,(No value)
213,(No value),2025-01-13 16:41:00,david.norris@eurofins-biomnis.ie,Pharma,biomnis.ie,(No value)
278,(No value),2024-05-02 10:04:00,monica@ridc.ie,Government Organisations,www.ridc.ie,(No value)
391,(No value),2025-04-09 11:43:00,tjcrowe@crowefarm.ie,Agriculture,crowefarm.ie,(No value)
470,(No value),2023-05-17 20:41:00,cbrosnan@masterlinklogistics.com,Logistics,masterlink.ie,(No value)
612,(No value),2021-04-15 11:04:00,oconnorm@ted.ie,Services / Gov,ted.ie,(No value)
822,(No value),2023-05-17 20:52:00,accountspayable@thehappypear.ie,Retail,thehappypear.ie,(No value)
1026,(No value),2023-10-24 13:11:00,mary.doolan@spk.com,Pharma,tandempm.ie,(No value)
1100,(No value),2021-01-25 19:09:00,darren.nutter@crs.ie,Baked Goods,crs.ie,(No value)


In [10]:
# Replace (No value) with Ireland in the filtered dataset
filtered_df.loc[
    filtered_df['Country/Region'] == '(No value)', 
    'Country/Region'
] = 'Ireland'

In [11]:
filtered_df['Country/Region'].value_counts()

Country/Region
Ireland    13877
Name: count, dtype: int64

In [12]:
personal_domains = [
    'gmail.com',
    'yahoo.com',
    'hotmail.com',
    'outlook.com',
    'live.com',
    'icloud.com'
]

# Removing personal emails that ALSO have no website

clean_df = filtered_df[
    ~(
        filtered_df['Email'].str.endswith(tuple(personal_domains), na=False) &
        (filtered_df['Website URL'] == '(No value)')
    )
]

In [13]:
clean_df.shape

(11820, 6)

In [14]:
clean_df[
    clean_df['Email'].str.endswith(tuple(personal_domains), na=False)
].head(10)

,Full Name,Create Date,Email,Industry,Website URL,Country/Region
2604,(No value),2023-05-17 20:52:00,filligans@gmail.com,Fruit and Veg,filligans.ie,Ireland
4057,(No value),2021-08-23 11:21:00,dawidmaruszak@gmail.com,(No value),mastermediauk.com,Ireland
4108,(No value),2025-04-04 10:14:00,alen.loncarek@gmail.com,Food,adriaticgroup.com,Ireland
4376,(No value),2020-02-25 12:41:00,efadailis@gmail.com,Services / Gov,mu.ie,Ireland
5349,(No value),2021-08-09 14:44:00,donaghyleisureltd@gmail.com,Hospitality,dnata.com,Ireland
6315,(No value),2023-04-12 09:21:00,liamshepp@gmail.com,Other,teagasc.ie,Ireland
8395,(No value),2022-11-03 10:09:00,newlines1153@gmail.com,Retail,supervalu.ie,Ireland
12010,(No value),2020-04-27 08:15:00,mclaughlins42@gmail.com,Beverages,7thraven.ie,Ireland
13726,(No value),2021-02-12 12:41:00,maherpauline2@gmail.com,Dairy,knockdrinna.com,Ireland
15617,(No value),2024-09-26 16:46:00,patennis86@hotmail.com,Pharma,irishequinecentre.ie,Ireland


In [15]:
clean_df['Website URL'] = (
    clean_df['Website URL']
    .str.strip()          # remove spaces
    .str.lower()          # standardise casing
)

In [16]:
clean_df['Website URL'].head(20)

10                    (no value)
19             strandlimerick.ie
28                    (no value)
40                    (no value)
43              cmsmarketing.com
52                    (no value)
55                    (no value)
57              abpfoodgroup.com
58                     kerry.com
62     kilbegganorganicfoods.com
63                  mossfield.ie
66               collarofgold.ie
68           lakelanddairies.com
77                      kerry.ie
78                    eirdata.ie
83                       wcdp.ie
90                    (no value)
94                    lecreme.ie
98              omega-pharma.com
103                   (no value)
Name: Website URL, dtype: str

In [17]:
# Filling missing Website URL using email domain

clean_df.loc[
    clean_df['Website URL'].isna(),
    'Website URL'
] = clean_df.loc[
    clean_df['Website URL'].isna(),
    'Email'
].str.split('@').str[1]

In [18]:
clean_df[clean_df['Website URL'].isna()].shape

(0, 6)

In [19]:
clean_df['Website URL'].sample(20)

62174                  quigleys.ie
51310     athlone-laboratories.com
10166                  brackens.ie
36466                       iol.ie
13366                   (no value)
13497               taratowers.com
32243                  scichem.com
25039          waltontransport.com
62689                   (no value)
22343                   www.rkd.ie
27051          rosaleenskitchen.ie
62       kilbegganorganicfoods.com
47096          monkstownflowers.ie
33126            redtorchginger.ie
43819                     bsci.com
23668                   (no value)
19831              farrellfoods.ie
2032                      walls.ie
61052                applegreen.ie
23026                      sisk.ie
Name: Website URL, dtype: str

In [20]:
clean_df = clean_df.dropna(subset=['Website URL'])

clean_df.shape

(11820, 6)

In [21]:
# Count duplicates based on Website URL
clean_df.duplicated(subset='Website URL').sum()

np.int64(6788)

In [22]:
# View duplicate companies
clean_df[
    clean_df.duplicated(subset='Website URL', keep=False)
].sort_values('Website URL').head(20)

,Full Name,Create Date,Email,Industry,Website URL,Country/Region
10,(No value),2025-07-04 10:28:00,william.doherty@abbott.com,Pharma,(no value),Ireland
36062,(No value),2023-04-26 09:20:00,terry.odwyer@crossrentalservices.com,Temperature Control,(no value),Ireland
36066,(No value),2023-04-26 09:20:00,terry.odwyer@crossrentalservices.com,Temperature Control,(no value),Ireland
36081,(No value),2022-03-29 12:27:00,dowling.colette@yahoo.ie,(No value),(no value),Ireland
36083,(No value),2023-05-17 20:29:00,luke.doyle@aryzta.com,Baked Goods,(no value),Ireland
36104,(No value),2022-03-29 12:27:00,richard@tastecounts.com,(No value),(no value),Ireland
36114,(No value),2023-05-17 20:29:00,john.buckley@clona.ie,Dairy,(no value),Ireland
36128,(No value),2023-04-26 16:09:00,daniel.oconnell@donnellyfresh.ie,Fruit and Veg,(no value),Ireland
36136,(No value),2023-04-26 09:20:00,terry.odwyer@crossrentalservices.com,Temperature Control,(no value),Ireland
36141,(No value),2023-04-26 09:20:00,terry.odwyer@crossrentalservices.com,Temperature Control,(no value),Ireland


In [23]:
# Standardise ALL "no value" variations first
clean_df['Website URL'] = (
    clean_df['Website URL']
    .str.strip()
    .str.lower()
    .replace('(no value)', pd.NA)
)

In [24]:
clean_df.loc[
    clean_df['Website URL'].isna(),
    'Website URL'
] = clean_df.loc[
    clean_df['Website URL'].isna(),
    'Email'
].str.split('@').str[1]

In [25]:
clean_df['Website URL'].isna().sum()

np.int64(0)

In [26]:
clean_df.duplicated(subset='Website URL').sum()

clean_df[
    clean_df.duplicated(subset='Website URL', keep=False)
].sort_values('Website URL').head(20)

,Full Name,Create Date,Email,Industry,Website URL,Country/Region
52289,(No value),2021-05-27 08:32:00,info@2getherstudios.ie,Media Production,2getherstudios.ie,Ireland
58198,(No value),2020-03-31 08:20:00,wes@2getherstudios.ie,Media Production,2getherstudios.ie,Ireland
41111,(No value),2022-08-10 09:48:00,ryan.oconnor@2sfg.com,Meat,2sfg.com,Ireland
62942,(No value),2020-03-31 08:20:00,padraic.marren@2sfg.com,Meat,2sfg.com,Ireland
13417,(No value),2020-03-31 08:20:00,pat.herbert@2sfg.com,Meat,2sfg.com,Ireland
20318,(No value),2020-03-31 08:20:00,declan.loftus@2sfg.com,Meat,2sfg.com,Ireland
32598,(No value),2020-03-31 08:20:00,tom.cronin@2sfg.com,Meat,2sfg.com,Ireland
4793,(No value),2020-03-31 08:20:00,raymond.farrelly@2sfg.com,Food,2sfg.com,Ireland
31884,(No value),2020-03-31 08:20:00,gareth.doherty@2sfg.com,Food,2sfg.com,Ireland
19843,(No value),2020-03-31 08:20:00,adrian.oconnor@2sfg.com,Meat,2sfg.com,Ireland


In [27]:
dedup_df = clean_df.drop_duplicates(subset='Email')

In [28]:
dedup_df.shape

(10108, 6)

In [29]:
dedup_df['Email'].duplicated().sum()

np.int64(0)

In [30]:
dedup_df.to_csv("clean_ireland_contacts.csv", index=False)